# 2 phase
- pre-training
- finetuning

## BERT 
- specialize in masked word prediction
- text classifications
- sentiment prediction
- document categorization
## GPT
- generative tasks

# Stages of building
- data prep and sampling
- attention mechanism
- llm architecture
- training loop
- model eval
- load pretrained weights
- finetuning
    - classifier (dataset with class labels)
    - personal assistant (instruct dataset)

## Data prep and sampling

- we work with word embeddings , eg: word2vec

- llm generally produce their own embeddings that are part of the input layer and are updated during training

- embedding size: dimentionality of the model's hidden states

### tokenizing text

In [2]:
with open("./ds.txt",'r',encoding="utf-8") as f:
    raw_text=f.read()

In [3]:
len(raw_text)

20480

In [4]:
raw_text[:99]

'I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no '

> vocabulary: how we map each unique word / character to a unique integer

In [5]:
import re

In [6]:
preprocessed= re.split(r'([,.?_!"()\']|--|\s)', raw_text)

In [7]:
preprocessed=[item.strip() for item in preprocessed if item.strip()]

In [9]:
# number of tokens
len(preprocessed)

4649

In [10]:
preprocessed[:30]

['I',
 'HAD',
 'always',
 'thought',
 'Jack',
 'Gisburn',
 'rather',
 'a',
 'cheap',
 'genius',
 '--',
 'though',
 'a',
 'good',
 'fellow',
 'enough',
 '--',
 'so',
 'it',
 'was',
 'no',
 'great',
 'surprise',
 'to',
 'me',
 'to',
 'hear',
 'that',
 ',',
 'in']

In [11]:
all_words=sorted(list(set(preprocessed)))

In [12]:
vocab_size=len(all_words)

In [13]:
vocab_size

1159

In [16]:
vocab={token:integer for integer,token in enumerate(all_words)}

- handling unknown words and efficient subword tokenizations is achieved via - bpe - byte pair encoding

- tiktoken efficiently implements bpe in rust

In [18]:
import tiktoken

In [19]:
tokenizer=tiktoken.get_encoding("gpt2")

In [22]:

text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
     "of someunknownPlace."
)

integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})

print(integers)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 1659, 617, 34680, 27271, 13]


In [26]:
tokenizer.encode("<|endoftext|>",allowed_special={"<|endoftext|>"})

[50256]

In [27]:
tokenizer.decode(integers)

'Hello, do you like tea? <|endoftext|> In the sunlit terracesof someunknownPlace.'

In [31]:
# this is the vocab size
tokenizer.n_vocab

50257

> bpe: freq characters into subwords and freq subwords into words

### sampling and sliding window

In [6]:
import torch
from torch.utils.data import Dataset, DataLoader
import tiktoken

In [7]:
class GPTDatasetV1(Dataset):
    def __init__(self,txt, tokenizer, max_length, stride):
        # txt-> total text to build the dataset from
        # tokenizer-> tokenzier to use
        # max_length-> max context length
        # stride-> sliding window size, how many tokens to slide for the new window
        self.tokenizer=tokenizer
        self.input_ids=[]
        self.target_ids=[]
        token_ids=tokenizer.encode(txt)
        for i in range(0, len(token_ids)-max_length, stride):
            input_chunk=token_ids[i:i+max_length]
            target_chunk=token_ids[i+1:i+max_length+1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __getitem__(self,idx):
        return self.input_ids[idx],self.target_ids[idx]
    def __len__(self):
        return len(self.input_ids)

In [8]:
def create_dataloaderv1(txt, batch_size=4,max_length=256,stride=128,shuffle=False,drop_last=True):
    tokenizer=tiktoken.get_encoding("gpt2")
    dataset=GPTDatasetV1(txt,tokenizer,max_length,stride)
    dataloader=DataLoader(dataset, batch_size=batch_size, shuffle=shuffle,drop_last=drop_last)
    return dataloader

In [9]:
with open("ds.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

In [10]:
dataloaderv1=create_dataloaderv1(raw_text,batch_size=1,max_length=4,stride=1, shuffle=False,drop_last=True)

In [11]:
data_iter=iter(dataloaderv1)
first_batch=next(data_iter)
print(first_batch)

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


In [12]:
len(first_batch[0]),len(first_batch[1])

(1, 1)

In [13]:
next(data_iter)

[tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]

In [14]:
dataloader = create_dataloaderv1(raw_text, batch_size=8, max_length=4, stride=4, shuffle=False)

In [15]:
next(iter(dataloader))

[tensor([[   40,   367,  2885,  1464],
         [ 1807,  3619,   402,   271],
         [10899,  2138,   257,  7026],
         [15632,   438,  2016,   257],
         [  922,  5891,  1576,   438],
         [  568,   340,   373,   645],
         [ 1049,  5975,   284,   502],
         [  284,  3285,   326,    11]]),
 tensor([[  367,  2885,  1464,  1807],
         [ 3619,   402,   271, 10899],
         [ 2138,   257,  7026, 15632],
         [  438,  2016,   257,   922],
         [ 5891,  1576,   438,   568],
         [  340,   373,   645,  1049],
         [ 5975,   284,   502,   284],
         [ 3285,   326,    11,   287]])]

In [16]:
len(dataloader)

160

In [17]:
next(iter(dataloader))

[tensor([[   40,   367,  2885,  1464],
         [ 1807,  3619,   402,   271],
         [10899,  2138,   257,  7026],
         [15632,   438,  2016,   257],
         [  922,  5891,  1576,   438],
         [  568,   340,   373,   645],
         [ 1049,  5975,   284,   502],
         [  284,  3285,   326,    11]]),
 tensor([[  367,  2885,  1464,  1807],
         [ 3619,   402,   271, 10899],
         [ 2138,   257,  7026, 15632],
         [  438,  2016,   257,   922],
         [ 5891,  1576,   438,   568],
         [  340,   373,   645,  1049],
         [ 5975,   284,   502,   284],
         [ 3285,   326,    11,   287]])]

In [18]:
len(dataloader)

160

### token embeddings

positional embeddings
- relative  (can generalize better to seq of varying lengths)
- absolute

In [20]:
tokenizer=tiktoken.get_encoding("gpt2")
tokenizer.n_vocab

50257

In [21]:
output_dim=256
vocab_size=50257
token_embedding_layer=torch.nn.Embedding(vocab_size,output_dim)

In [22]:
max_length = 4
dataloader = create_dataloaderv1(
    raw_text, batch_size=8, max_length=max_length,
    stride=max_length, shuffle=False
)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)

In [23]:

print("Token IDs:\n", inputs)
print("\nInputs shape:\n", inputs.shape)

Token IDs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Inputs shape:
 torch.Size([8, 4])


In [24]:
token_embeddings=token_embedding_layer(inputs)
print(token_embeddings.shape)

torch.Size([8, 4, 256])


In [25]:
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)

In [27]:

pos_embeddings = pos_embedding_layer(torch.arange(max_length))
print(pos_embeddings.shape)

torch.Size([4, 256])


In [28]:
input_embeddings = token_embeddings + pos_embeddings
print(input_embeddings.shape)

torch.Size([8, 4, 256])


## attention mechanisms

progress
- simplified self attention
- self-attention
- causal attention
- multi-head attention

> before llm, RNN were popular encoder-decoder architecture for language translation
> rnn is a type of nn where outputs form previous steps are fed as inputs to the current step
> good for seq data like text

### RNN primer
- in an encoder-decoder rnn, input text is fed into the encoder
- encoder processes this sequentially
- encoder updates its hidden state (internal values at the hidden layers) at each step trying
    to capture the entire meaning of the input sequence in the final hidden state
- decoder takes the final hidden state to start generating the translated sentence
- one word at a  time
- also updates its hidden state at each step which is supposed to carry the context necessary for
    the next-word
---
### drawback
- encoder-decoder RNN cant directly access earlier hidden states from the encoder during the
decoding phase
- it relies on the current hidden state leads to loss of context in complex sentences
where dependencies might span longg distances

## simple self attn without trainable weights

- context vector for each input element: combines information from all other 
input elements.
- computed using the attn weigths. attn weights are calculated wrt to input element and all other elements

In [3]:
import torch
inputs = torch.tensor(
[[0.43, 0.15, 0.89], # Your (x^1)
[0.55, 0.87, 0.66], # journey (x^2)
[0.57, 0.85, 0.64], # starts (x^3)
[0.22, 0.58, 0.33], # with (x^4)
[0.77, 0.25, 0.10], # one (x^5)
[0.05, 0.80, 0.55]] # step (x^6)
)

#### step1: calculate attention scores
- calcualted between a query and all other input elements as dot product

> attn scores are just dot prod of embeddings

In [4]:
inputs.shape[0]

6

In [5]:
query=inputs[1]
attn_score_2=torch.empty(inputs.shape[0])

In [6]:
attn_score_2

tensor([0., 0., 0., 0., 0., 0.])

In [7]:
for i,x_i in enumerate(inputs):
    attn_score_2[i]=torch.dot(query, x_i)
print(attn_score_2)

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


> dot prod determines the extent to which elements in a sequence attend to each other

#### step2: normalize the attn scores

> goal: after normalizing the attn scores,we get attn weights that sum upto one

In [8]:
attn_weights_2_tmp=attn_score_2/attn_score_2.sum()

In [9]:
print(attn_weights_2_tmp)

tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])


In [10]:
sum(attn_weights_2_tmp)

tensor(1.0000)

In [11]:
help(torch.softmax)

Help on built-in function softmax in module torch:

softmax(...)
    softmax(input, dim, *, dtype=None) -> Tensor

    Alias for :func:`torch.nn.functional.softmax`.



> use softmax for normalization, it ensures weights are always positive

In [12]:
attn_weights_2=torch.softmax(attn_score_2,dim=0)

#### step3: calculating the context vector

In [13]:
query=inputs[1]
context_vec_2=torch.zeros(query.shape)
for i,x_i in enumerate(inputs):
    context_vec_2+=attn_weights_2[i]*x_i
print(context_vec_2)

tensor([0.4419, 0.6515, 0.5683])


#### calulating attn weights for all inputs

In [14]:
inputs

tensor([[0.4300, 0.1500, 0.8900],
        [0.5500, 0.8700, 0.6600],
        [0.5700, 0.8500, 0.6400],
        [0.2200, 0.5800, 0.3300],
        [0.7700, 0.2500, 0.1000],
        [0.0500, 0.8000, 0.5500]])

In [15]:
# attn scores
attn_scores=inputs@inputs.T

In [16]:
attn_scores

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])

In [17]:
attn_scores[0,1],attn_scores[1,0]

(tensor(0.9544), tensor(0.9544))

In [18]:
# attn weigths - normalizing
attn_weights=torch.softmax(attn_scores,dim=1)

In [19]:
attn_weights

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])

In [20]:
attn_weights.sum(dim=1)

tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])

In [21]:
inputs.shape, attn_weights.shape

(torch.Size([6, 3]), torch.Size([6, 6]))

In [22]:
# context vectors
context_vec=attn_weights@inputs

In [23]:
context_vec

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])

#### sanity check

In [62]:
attn_weights[1]

tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])

In [84]:
attn_w2=attn_weights[1]

In [85]:
attn_w2

tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])

In [80]:
attn_w2=attn_w2.unsqueeze(0)

In [81]:
attn_w2.shape

torch.Size([1, 6])

In [82]:
attn_w2

tensor([[0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581]])

In [65]:
inputs.shape

torch.Size([6, 3])

In [83]:
attn_w2@inputs

tensor([[0.4419, 0.6515, 0.5683]])

In [86]:
inputs[1]

tensor([0.5500, 0.8700, 0.6600])

## self-attention with trainable weights

- scaled dot-product attention
- Wq,Wk, Wv matrics used to project the embedded input tokens x(i) into query key and value vectors

In [24]:
x_2=inputs[1]
d_in=inputs.shape[1]
d_out=2
d_in,d_out

(3, 2)

In [25]:
torch.manual_seed(123)

In [29]:
W_query=torch.nn.Parameter(torch.rand(d_in,d_out),requires_grad=False)
W_key=torch.nn.Parameter(torch.rand(d_in,d_out),requires_grad=False)
W_value=torch.nn.Parameter(torch.rand(d_in,d_out),requires_grad=False)

In [30]:
query_2= x_2@W_query
key_2=x_2@W_key
value_2=x_2@W_value

In [31]:
query_2,key_2,value_2

(tensor([1.7386, 1.3068]), tensor([0.4395, 0.8128]), tensor([0.9292, 1.3662]))

In [32]:
# we requires keys and values for all input elements
# as they are involved in computing the attn weights wrt to x_2
keys=inputs@W_key
values=inputs@W_value

In [33]:
keys.shape, values.shape

(torch.Size([6, 2]), torch.Size([6, 2]))

### compute attn scores

In [38]:
keys_2=keys[1]

In [39]:
keys_2

tensor([0.4395, 0.8128])

In [40]:
attn_score_22=query_2.dot(keys_2)

In [41]:
attn_score_22

tensor(1.8263)

- computed using the dot prod of queries and keys

In [34]:
query_2.shape, keys.shape

(torch.Size([2]), torch.Size([6, 2]))

In [35]:
attn_scores_2=query_2 @ keys.T

In [36]:
attn_scores_2

tensor([1.2999, 1.8263, 1.7990, 1.0184, 0.8015, 1.3484])

### attn scores

In [42]:
d_k=keys.shape[-1]
d_k

2

In [45]:
# The reason for the normalization by the embedding dimension size is to
# improve the training performance by avoiding small gradients. For instance,
# when scaling up the embedding dimension, which is typically greater than
# thousand for GPT-like LLMs, large dot products can result in very small
# gradients during backpropagation due to the softmax function applied to
# them. As dot products increase, the softmax function behaves more like a
# step function, resulting in gradients nearing zero. These small gradients can
# drastically slow down learning or cause training to stagnate.

In [43]:
attn_weights_2= torch.softmax(attn_scores_2/d_k**0.5,dim=-1)

In [47]:
attn_weights_2, attn_weights_2.shape

(tensor([0.1555, 0.2256, 0.2213, 0.1274, 0.1093, 0.1609]), torch.Size([6]))

In [48]:
values.shape

torch.Size([6, 2])

In [49]:
context_vec_2=attn_weights_2@values

In [50]:
context_vec_2

tensor([0.7332, 1.0848])

> query , key , value => store, search, retrieve

```The terms "key," "query," and "value" in the context of attention mechanisms
are borrowed from the domain of information retrieval and databases, where
similar concepts are used to store, search, and retrieve information.
A "query" is analogous to a search query in a database. It represents the
current item (e.g., a word or token in a sentence) the model focuses on or
tries to understand. The query is used to probe the other parts of the input
sequence to determine how much attention to pay to them.
The "key" is like a database key used for indexing and searching. In the
attention mechanism, each item in the input sequence (e.g., each word in a
sentence) has an associated key. These keys are used to match with the query.
The "value" in this context is similar to the value in a key-value pair in a
database. It represents the actual content or representation of the input items.
Once the model determines which keys (and thus which parts of the input)
are most relevant to the query (the current focus item), it retrieves the
corresponding values.
```

### compact self attention impl

In [1]:
import torch.nn as nn

In [58]:
class SelfAttention(nn.Module):
    def __init__(self,d_in,d_out, qkv_bias=False):
        super().__init__()
        self.d_out=d_out
        self.W_query=nn.Linear(d_in,d_out,bias=qkv_bias)
        self.W_key=nn.Linear(d_in,d_out,bias=qkv_bias)
        self.W_value=nn.Linear(d_in,d_out,bias=qkv_bias)

    def forward(self,x):
        queries=self.W_query(x)
        keys=self.W_key(x)
        values=self.W_value(x)
        attn_scores=queries@keys.T
        # normalize over the keys, i.e dim=-1 last dimension
        attn_weights=torch.softmax(attn_scores/keys.shape[-1]**0.5,dim=-1)
        context_vec=attn_weights@values
        return context_vec

In [59]:
torch.manual_seed(789)
sa_v2 = SelfAttention(3, 2)
print(sa_v2(inputs))

tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)


## Causal self attention

In [71]:
class CausalSelfAttention(nn.Module):
    def __init__(self, d_in,d_out, context_length,dropout,qkv_bias=False):
        super().__init__()
        self.W_query,self.W_key,self.W_value=nn.Linear(d_in,d_out,bias=qkv_bias),nn.Linear(d_in,d_out,bias=qkv_bias),nn.Linear(d_in,d_out,bias=qkv_bias)
        self.d_out=d_out
        self.dropout=nn.Dropout(dropout)
        self.register_buffer("mask",
        torch.triu(torch.ones(context_length, context_length),
        diagonal=1))
    def forward(self,x):
        b,num_tokens,d_in=x.shape
        queries,keys,values=self.W_query(x),self.W_key(x),self.W_value(x)
        # keys -> b, num_tokens, d_out
        # queries -> b, num_tokens, d_out
        # queries -> b, d_out, num_tokens
        attn_scores= queries@keys.transpose(1,2)
        attn_scores.masked_fill_(
            self.mask.bool()[:num_tokens,:num_tokens],-torch.inf
        )
        attn_weights=torch.softmax(attn_scores/keys.shape[-1]**0.5,dim=-1)
        context_vec=attn_weights@values
        return context_vec

In [72]:
batch = torch.stack((inputs, inputs), dim=0)
print(batch.shape) #A

torch.Size([2, 6, 3])


In [73]:
torch.manual_seed(123)
context_length = batch.shape[1]
ca = CausalSelfAttention(d_in, d_out, context_length, 0.0)
context_vecs = ca(batch)
print("context_vecs.shape:", context_vecs.shape)

context_vecs.shape: torch.Size([2, 6, 2])


In [74]:
context_vecs

tensor([[[-0.4519,  0.2216],
         [-0.5874,  0.0058],
         [-0.6300, -0.0632],
         [-0.5675, -0.0843],
         [-0.5526, -0.0981],
         [-0.5299, -0.1081]],

        [[-0.4519,  0.2216],
         [-0.5874,  0.0058],
         [-0.6300, -0.0632],
         [-0.5675, -0.0843],
         [-0.5526, -0.0981],
         [-0.5299, -0.1081]]], grad_fn=<UnsafeViewBackward0>)

## multihead attention


In [2]:

import torch
inputs = torch.tensor(
[[0.43, 0.15, 0.89], # Your (x^1)
[0.55, 0.87, 0.66], # journey (x^2)
[0.57, 0.85, 0.64], # starts (x^3)
[0.22, 0.58, 0.33], # with (x^4)
[0.77, 0.25, 0.10], # one (x^5)
[0.05, 0.80, 0.55]] # step (x^6)
)

In [3]:
inputs.shape

torch.Size([6, 3])

In [4]:
batch = torch.stack((inputs, inputs), dim=0)
print(batch.shape) #A

torch.Size([2, 6, 3])


## explanation
Every single head sees the entire input sequence (all the tokens). They do not look at just a slice of the sentence.

What the heads split up is the embedding dimension (the features), not the number of tokens (the words).

Let's break down exactly why this happens and where the split actually occurs.

The Sequence vs. The Features
Think about the shape of your tensor right before you calculate the attention scores:
query shape: (batch_size, num_heads, num_tokens, head_dim)
In your example numbers: (2, 8, 16, 32)

Let's isolate just one sequence in the batch and look at what Head 1 gets:

Head 1's slice: A matrix of 16 (tokens) by 32 (features).

Notice that the 16 is completely intact. Head 1 has access to Token 1, Token 2, Token 3... all the way to Token 16. It sees the whole sentence.

What it doesn't have the full picture of is the d_out dimension. It only gets 32 out of the 256 total features available for those words.

Proof in the Attention Scores
You can prove that every head sees every token by looking at your scaled dot-product calculation:

Python
attn_scores = query @ keys.transpose(2,3)
The resulting attn_scores tensor has the shape (batch_size, num_heads, num_tokens, num_tokens). Using your numbers, that is (2, 8, 16, 16).

That 16 x 16 grid exists inside every single one of the 8 heads. That means inside Head 1, Token 1 calculates an attention score with Token 1, Token 2, Token 3, etc. Every token interacts with every other token, inside every single head.

The "Inspectors" Analogy
If it helps to visualize it, imagine your input sequence is a line of 16 cars.

The Embedding (d_out): A massive, 256-page spec sheet for each car (paint color, engine type, tire pressure, top speed, etc.).

Single-Head Attention: One general inspector looks at all 16 cars and tries to read the entire 256-page spec sheet for every car at the same time to figure out how the cars relate to each other. It's overwhelming, and they average things out.

Multi-Head Attention (8 Heads): You hire 8 highly specialized inspectors.

Inspector 1 (Head 1) looks at all 16 cars, but only cares about the paint and interior colors (features 0-31).

Inspector 2 (Head 2) looks at all 16 cars, but only cares about the engine and transmission (features 32-63).

Why combine them at the end?
As you noted, we eventually combine them using .contiguous().view(...) and self.out_proj.

We do this because while Inspector 1 perfectly understands which cars have matching colors, and Inspector 2 perfectly understands which cars can race each other, neither of them has the full picture of the cars. The final linear projection layer is the "meeting room" where all 8 inspectors bring their 16 x 32 reports, staple them back together into a 16 x 256 report, and synthesize a final understanding of the entire sequence.

In [6]:
class MultiHeadAttention(nn.Module):
    def __init__(self,d_in,d_out, context_length,dropout,num_heads,qkv_bias=False):
        super().__init__()
        assert (d_out%num_heads==0),"d_out must be divisible by num_heads"
        self.d_out=d_out
        self.num_heads=num_heads
        self.head_dim=d_out//num_heads
        self.W_query=nn.Linear(d_in,d_out,bias=qkv_bias)
        self.W_key=nn.Linear(d_in,d_out,bias=qkv_bias)
        self.W_value=nn.Linear(d_in,d_out,bias=qkv_bias)
        self.out_proj=nn.Linear(d_out,d_out) #linear layer to combine head outputs
        self.dropout=nn.Dropout(dropout)
        self.register_buffer("mask",torch.triu(torch.ones(context_length,context_length),diagonal=1))

    def forward(self,x):
        # input is of format batch_size, num_tokens, d_in (embed dimension)
        batch_size,num_tokens,d_in=x.shape
        # calculating
        # query
        # size: b_size, num_tokens, d_out
        query=self.W_query(x);
        # keys
        # size: b_size, num_tokens, d_out
        keys=self.W_key(x);
        # values
        # size: b_size, num_tokens, d_out
        values=self.W_value(x);
        # splitting each of query key value in to multiple heads
        # b_size, num_tokens, d_out -> b_size, num_tokens, num_heads, head_dim
        # why cant we just do query.view(batch_size,num_tokens,self.head_dim,self.num_heads) -- llm clarify 
        query=query.view(batch_size,num_tokens,self.num_heads,self.head_dim)
        # b_size, num_tokens, d_out -> b_size, num_tokens, num_heads, head_dim
        keys=keys.view(batch_size,num_tokens,self.num_heads,self.head_dim)
        # b_size, num_tokens, d_out -> b_size, num_tokens, num_heads, head_dim
        values=values.view(batch_size,num_tokens,self.num_heads,self.head_dim)
        # transposing the resultant tensors so that
        # we get the head dimension as the second dimension and 
        # number of tokens as the third dimension
        # b_size, num_tokens, num_heads, head_dim -> b_size, num_heads,num_tokens, head_dim
        query=query.transpose(1,2)
        keys=keys.transpose(1,2)
        values=values.transpose(1,2)
         # explanation of the mechanism:
        # batch_size,number_token, d_in (or embed dimension)
        # let the input be 2,16, 512
        # number of heads be 8
        # query's size before the next step will be-> 2, 16, 8,32
        # a way of interpreting this is, for the batch we have, there are 2 input text or sentences each has 16 tokens
        # now for each sentence we have 8 heads, each head has 32 dimension
        # performing scaled dot product attention
        # attn_scores shape -> batch_size,num_heads,num_tokens,num_tokens
        attn_scores=query@keys.transpose(2,3)
        #apply mask
        mask_bool=self.mask.bool()[:num_tokens,:num_tokens]
        #attn weights
        # shape; batch_size,num_heads,num_tokens,num_tokens
        attn_scores.masked_fill_(mask_bool,-torch.inf)
        attn_weights=torch.softmax(attn_scores/keys.shape[-1]**0.5,dim=-1)
        attn_weights=self.dropout(attn_weights)
        # context vector
        # shape before transpose: batch_size, num_heads,num_tokens, head_dim
        # after transpose: batch_size, num_tokens,num_heads, head_dim
        context_vec=(attn_weights @ values).transpose(1,2)
        # combine heads and projection
        # shape= batch_size,num_tokens,d_out
        context_vec=context_vec.contiguous().view(batch_size,num_tokens,self.d_out)
        context_vec=self.out_proj(context_vec)
        return context_vec

In [7]:
torch.manual_seed(123)

batch_size, context_length, d_in = batch.shape
d_out = 2
mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads=2)

context_vecs = mha(batch)

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

tensor([[[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]],

        [[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]]], grad_fn=<ViewBackward0>)
context_vecs.shape: torch.Size([2, 6, 2])


#  Implementing a GPT model from Scratch To Generate Text

In [6]:
import torch

In [7]:
GPT_CONFIG_124M = {
"vocab_size": 50257, # Vocabulary size
"context_length": 1024,
 # Context length
"emb_dim": 768,
 # Embedding dimension
"n_heads": 12,
 # Number of attention heads
"n_layers": 12,
 # Number of layers
"drop_rate": 0.1,
 # Dropout rate
"qkv_bias": False
 # Query-Key-Value bias
}

## layer normalization
- adjust the activations of a neural network layer to have a mean of 0 and a variance of 1
- speeds up the convergence to effective weights and ensures consistent
reliable training
- applied before multi-head attention module and before final output layer

In [8]:
import torch.nn as nn

### layer norm vs batch norm
- batch norm: norm across the batch dimension
- layer norm: norm across the feat dimension
- layer norm normalizes each input independent of the batch size, offers more flexibility
and stability in these scenarios

In [9]:
class LayerNorm(nn.Module):
    def __init__(self,emb_dim):
        super().__init__()
        self.eps=1e-5
        # eps is added to vairance to prevent divison by zero during normalization
        
        self.scale=nn.Parameter(torch.ones(emb_dim))
        self.shift=nn.Parameter(torch.zeros(emb_dim))
        # scale and shift are two trainable params
    def forward(self,x):
        mean=x.mean(dim=-1,keepdim=True)
        var=x.var(dim=-1,keepdim=True,unbiased=False)
        norm_x=(x-mean)/torch.sqrt(var+self.eps)
        return self.scale*norm_x+self.shift
        

## feed forward and gelu activation

### gelu activations

In [10]:
class GELU(nn.Module):
    def __init__(self):
        super().__init__()
    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
        torch.sqrt(torch.tensor(2.0 / torch.pi)) *
        (x + 0.044715 * torch.pow(x, 3))
        ))

> the FeedForward module is a small
neural network consisting of two Linear layers and a GELU activation
function. In the 124 million parameter GPT model, it receives the input
batches with tokens that have an embedding size of 768 each via the
GPT_CONFIG_124M dictionary where GPT_CONFIG_124M["emb_dim"] = 768

> The FeedForward module we implemented in this section plays a crucial role
in enhancing the model's ability to learn from and generalize the data.
Although the input and output dimensions of this module are the same, it
internally expands the embedding dimension into a higher-dimensional space
through the first linear layer as illustrated in Figure 4.10. This expansion is
followed by a non-linear GELU activation, and then a contraction back to the
original dimension with the second linear transformation. Such a design
allows for the exploration of a richer representation space

In [11]:
class FeedForward(nn.Module):
    def __init__(self,cfg):
        super().__init__()
        self.layers=nn.Sequential(nn.Linear(cfg['emb_dim'],4*cfg['emb_dim']),
                                 GELU(),
                                 nn.Linear(4*cfg['emb_dim'],cfg['emb_dim']),)
    def forward(self,x):
        return self.layers(x)

## shortcut connections/skip/residual connections

- play a crucial role in preserving the flow of gradients during the backward pass in training
- In conclusion, shortcut connections are important for overcoming the
limitations posed by the vanishing gradient problem in deep neural networks.
Shortcut connections are a core building block of very large models such as
LLMs, and they will help facilitate more effective training by ensuring
consistent gradient flow across layers when we train the GPT model in the
next chapter.

## 
final transformer block

In [12]:
class MultiHeadAttention(nn.Module):
    def __init__(self,d_in,d_out, context_length,dropout,num_heads,qkv_bias=False):
        super().__init__()
        assert (d_out%num_heads==0),"d_out must be divisible by num_heads"
        self.d_out=d_out
        self.num_heads=num_heads
        self.head_dim=d_out//num_heads
        self.W_query=nn.Linear(d_in,d_out,bias=qkv_bias)
        self.W_key=nn.Linear(d_in,d_out,bias=qkv_bias)
        self.W_value=nn.Linear(d_in,d_out,bias=qkv_bias)
        self.out_proj=nn.Linear(d_out,d_out) #linear layer to combine head outputs
        self.dropout=nn.Dropout(dropout)
        self.register_buffer("mask",torch.triu(torch.ones(context_length,context_length),diagonal=1))

    def forward(self,x):
        # input is of format batch_size, num_tokens, d_in (embed dimension)
        batch_size,num_tokens,d_in=x.shape
        # calculating
        # query
        # size: b_size, num_tokens, d_out
        query=self.W_query(x);
        # keys
        # size: b_size, num_tokens, d_out
        keys=self.W_key(x);
        # values
        # size: b_size, num_tokens, d_out
        values=self.W_value(x);
        # splitting each of query key value in to multiple heads
        # b_size, num_tokens, d_out -> b_size, num_tokens, num_heads, head_dim
        # why cant we just do query.view(batch_size,num_tokens,self.head_dim,self.num_heads) -- llm clarify 
        query=query.view(batch_size,num_tokens,self.num_heads,self.head_dim)
        # b_size, num_tokens, d_out -> b_size, num_tokens, num_heads, head_dim
        keys=keys.view(batch_size,num_tokens,self.num_heads,self.head_dim)
        # b_size, num_tokens, d_out -> b_size, num_tokens, num_heads, head_dim
        values=values.view(batch_size,num_tokens,self.num_heads,self.head_dim)
        # transposing the resultant tensors so that
        # we get the head dimension as the second dimension and 
        # number of tokens as the third dimension
        # b_size, num_tokens, num_heads, head_dim -> b_size, num_heads,num_tokens, head_dim
        query=query.transpose(1,2)
        keys=keys.transpose(1,2)
        values=values.transpose(1,2)
         # explanation of the mechanism:
        # batch_size,number_token, d_in (or embed dimension)
        # let the input be 2,16, 512
        # number of heads be 8
        # query's size before the next step will be-> 2, 16, 8,32
        # a way of interpreting this is, for the batch we have, there are 2 input text or sentences each has 16 tokens
        # now for each sentence we have 8 heads, each head has 32 dimension
        # performing scaled dot product attention
        # attn_scores shape -> batch_size,num_heads,num_tokens,num_tokens
        attn_scores=query@keys.transpose(2,3)
        #apply mask
        mask_bool=self.mask.bool()[:num_tokens,:num_tokens]
        #attn weights
        # shape; batch_size,num_heads,num_tokens,num_tokens
        attn_scores.masked_fill_(mask_bool,-torch.inf)
        attn_weights=torch.softmax(attn_scores/keys.shape[-1]**0.5,dim=-1)
        attn_weights=self.dropout(attn_weights)
        # context vector
        # shape before transpose: batch_size, num_heads,num_tokens, head_dim
        # after transpose: batch_size, num_tokens,num_heads, head_dim
        context_vec=(attn_weights @ values).transpose(1,2)
        # combine heads and projection
        # shape= batch_size,num_tokens,d_out
        context_vec=context_vec.contiguous().view(batch_size,num_tokens,self.d_out)
        context_vec=self.out_proj(context_vec)
        return context_vec

In [13]:
!ls

ds.txt	images	main.ipynb  main.py  pyproject.toml  README.md	uv.lock


In [14]:
!ls images

gpt_2_arch.png	transformer_block.png


![transformer block](./images/transformer_block.png)

> Layer normalization (LayerNorm) is applied before each of these two
components, and dropout is applied after them to regularize the model and
prevent overfitting. This is also known as Pre-LayerNorm. Older
architectures, such as the original transformer model, applied layer
normalization after the self-attention and feed-forward networks instead,
known as Post-LayerNorm, which often leads to worse training dynamics.

In [15]:
class TransformerBlock(nn.Module):
    def __init__(self,cfg):
        super().__init__()
        self.attn=MultiHeadAttention(d_in=cfg['emb_dim'],
                                    d_out=cfg['emb_dim'],
                                    context_length=cfg['context_length'],
                                    dropout=cfg['drop_rate'],
                                    num_heads=cfg['n_heads'],
                                    qkv_bias=cfg['qkv_bias'])
        self.ff=FeedForward(cfg)
        self.norm1=LayerNorm(cfg['emb_dim'])
        self.norm2=LayerNorm(cfg['emb_dim'])
        self.drop_resid=nn.Dropout(cfg['drop_rate'])
    def forward(self,x):
        shortcut=x
        x=self.norm1(x)
        x=self.attn(x)
        x=self.drop_resid(x)
        x=x+shortcut #skip conn

        shortcut=x

        x=self.norm2(x)
        x=self.ff(x)
        x=self.drop_resid(x)
        x=x+shortcut
        return x

>The preservation of shape throughout the transformer block architecture is
not incidental but a crucial aspect of its design. This design enables its
effective application across a wide range of sequence-to-sequence tasks,
where each output vector directly corresponds to an input vector, maintaining
a one-to-one relationship. However, the output is a context vector that
encapsulates information from the entire input sequence, as we learned in
chapter 3. This means that while the physical dimensions of the sequence
(length and feature size) remain unchanged as it passes through the
transformer block, the content of each output vector is re-encoded to integrate
contextual information from across the entire input sequence.

In [16]:
torch.manual_seed(123)
x = torch.rand(2, 4, 768) #A
block = TransformerBlock(GPT_CONFIG_124M)
output = block(x)
print("Input shape:", x.shape)
print("Output shape:", output.shape)
# The output is as follows:
# Input shape: torch.Size([2, 4, 768])
# Output shape: torch.Size([2, 4, 768])


Input shape: torch.Size([2, 4, 768])
Output shape: torch.Size([2, 4, 768])


## putting together the final gpt model

!['gpt 2 architecture'](./images/gpt_2_arch.png)

In [17]:
class GPTModel(nn.Module):
    def __init__(self,cfg):
        super().__init__()
        self.tok_emb=nn.Embedding(cfg['vocab_size'],cfg['emb_dim'])
        self.pos_emb=nn.Embedding(cfg['context_length'],cfg['emb_dim'])
        self.drop_emb=nn.Dropout(cfg['drop_rate'])

        self.trf_blocks=nn.Sequential(*[TransformerBlock(cfg) for _ in range(cfg['n_layers'])])
        self.final_norm=LayerNorm(cfg['emb_dim'])
        self.out_head=nn.Linear(cfg['emb_dim'],cfg['vocab_size'],bias=False)
    def forward(self,in_idx):
        batch_size,seq_len=in_idx.shape
        tok_embeds=self.tok_emb(in_idx)

        pos_embeds=self.pos_emb(torch.arange(seq_len,device=in_idx.device))
        x=tok_embeds+pos_embeds
        x=self.drop_emb(x)
        x=self.trf_blocks(x)
        x=self.final_norm(x)
        logits=self.out_head(x)
        return logits

In [18]:
inputs=torch.tensor([[ 6109,3626,6100,345],[ 6109,1110,6622,257]])

In [19]:
inputs.shape

torch.Size([2, 4])

In [20]:
torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
out = model(inputs)
print("Input batch:\n", inputs)
print("\nOutput shape:", out.shape)
print(out)

Input batch:
 tensor([[6109, 3626, 6100,  345],
        [6109, 1110, 6622,  257]])

Output shape: torch.Size([2, 4, 50257])
tensor([[[ 0.3613,  0.4222, -0.0711,  ...,  0.3483,  0.4661, -0.2838],
         [-0.1792, -0.5660, -0.9485,  ...,  0.0477,  0.5181, -0.3168],
         [ 0.7120,  0.0332,  0.1085,  ...,  0.1018, -0.4327, -0.2553],
         [-1.0076,  0.3418, -0.1190,  ...,  0.7195,  0.4023,  0.0532]],

        [[-0.2564,  0.0900,  0.0335,  ...,  0.2659,  0.4454, -0.6806],
         [ 0.1230,  0.3653, -0.2074,  ...,  0.7705,  0.2710,  0.2246],
         [ 1.0558,  1.0318, -0.2800,  ...,  0.6936,  0.3205, -0.3178],
         [-0.1565,  0.3926,  0.3288,  ...,  1.2630, -0.1858,  0.0388]]],
       grad_fn=<UnsafeViewBackward0>)


>a curious reader might notice a discrepancy. Earlier, we spoke of
initializing a 124 million parameter GPT model, so why is the actual number
of parameters 163 million, as shown in the preceding code output?
The reason is a concept called weight tying that is used in the original GPT-2
architecture, which means that the original GPT-2 architecture is reusing the
weights from the token embedding layer inits output layer

In [21]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Total number of parameters: {total_params:,}")


Total number of parameters: 163,009,536


In [22]:
print("Token embedding layer shape:", model.tok_emb.weight.shape)
print("Output layer shape:", model.out_head.weight.shape)

Token embedding layer shape: torch.Size([50257, 768])
Output layer shape: torch.Size([50257, 768])


### acoounting for weight tying

In [23]:
total_params = sum(p.numel() for p in model.parameters())
total_params-=sum(p.numel() for p in model.out_head.parameters())
print(f"Total number of parameters: {total_params:,}")


Total number of parameters: 124,412,160


In [26]:
dir(model)

['T_destination',
 '__annotate_func__',
 '__call__',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__firstlineno__',
 '__format__',
 '__ge__',
 '__getattr__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__setstate__',
 '__sizeof__',
 '__static_attributes__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_apply',
 '_backward_hooks',
 '_backward_pre_hooks',
 '_buffers',
 '_call_impl',
 '_compiled_call_impl',
 '_forward_hooks',
 '_forward_hooks_always_called',
 '_forward_hooks_with_kwargs',
 '_forward_pre_hooks',
 '_forward_pre_hooks_with_kwargs',
 '_get_backward_hooks',
 '_get_backward_pre_hooks',
 '_get_name',
 '_is_full_backward_hook',
 '_load_from_state_dict',
 '_load_state_dict_post_hooks',
 '_load_state_dict_pre_hooks',
 '_maybe_warn_non_full_backward_hook',
 '_modules',

## generating text

### greedy decoding

In [35]:
def generate_text_simple(model,idx,max_new_tokens,context_size):
    # idx -> (batch, n_tokens) array of indices in the current context
    for _ in range(max_new_tokens):
        # pick only last context_size tokens
        idx_cond=idx[:,-context_size:]
        with torch.no_grad():
            logits=model(idx_cond)
        logits=logits[:,-1,:] # batch,n_tokens,vocab_size -> batch,vocab_size
        probas=torch.softmax(logits,dim=-1) #(batch, vocab_size)
        idx_next=torch.argmax(probas,dim=-1,keepdim=True)# (batch,1)
        idx=torch.cat((idx,idx_next),dim=-1) #(batch, n_tokens+1)
    return idx

In [28]:
import torch

In [29]:
import tiktoken
tokenizer=tiktoken.get_encoding("gpt2")
start_context="Hello, I am"
encoded=tokenizer.encode(start_context)
print(f"{encoded=}")
encoded_tensor=torch.tensor(encoded).unsqueeze(0 )#adding a batch dim
print(f"{encoded_tensor.shape=}")


encoded=[15496, 11, 314, 716]
encoded_tensor.shape=torch.Size([1, 4])


In [30]:
model.eval()
out=generate_text_simple(model,idx=encoded_tensor,max_new_tokens=6,context_size=GPT_CONFIG_124M['context_length'])

In [33]:
print(f"{out=}")
print(f"{len(out[0])=}")

out=tensor([[15496,    11,   314,   716, 27018, 24086, 47843, 30961, 42348,  7267]])
len(out[0])=10


In [34]:
decoded_text=tokenizer.decode(out.squeeze(0).tolist())
print(decoded_text)

Hello, I am Featureiman Byeswickattribute argue


# Pretraiining on unlabelled data

In [37]:
import torch
# from chapter04 import GPTModel
GPT_CONFIG_124M = {
"vocab_size": 50257,
"context_length": 256, #A
"emb_dim": 768,
"n_heads": 12,
"n_layers": 12,
"drop_rate": 0.1,
 #B
"qkv_bias": False
}
torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
model.eval()

GPTModel(
  (tok_emb): Embedding(50257, 768)
  (pos_emb): Embedding(256, 768)
  (drop_emb): Dropout(p=0.1, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (attn): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=False)
        (W_key): Linear(in_features=768, out_features=768, bias=False)
        (W_value): Linear(in_features=768, out_features=768, bias=False)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_resid): Dropout(p=0.1, inplace=False)
    )
    (1): TransformerBlock(
      (attn): MultiHeadAttention(
        (W_query): Linear(in_features=

In [38]:
def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text, allowed_special={'<|endoftext|>'})
    encoded_tensor = torch.tensor(encoded).unsqueeze(0) # add batch dimension
    return encoded_tensor

def token_ids_to_text(token_ids, tokenizer):
    flat = token_ids.squeeze(0) # remove batch dimension
    return tokenizer.decode(flat.tolist())

start_context = "Every effort moves you"
tokenizer = tiktoken.get_encoding("gpt2")

token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids(start_context, tokenizer),
    max_new_tokens=10,
    context_size=GPT_CONFIG_124M["context_length"]
)

print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

Output text:
 Every effort moves you rentingetic wasnم refres RexMeCHicular stren


## calculating the text generation loss

In [39]:
inputs = torch.tensor([[16833, 3626, 6100],   # ["every effort moves",
                       [40,    1107, 588]])   #  "I really like"]

targets = torch.tensor([[3626, 6100, 345  ],  # [" effort moves you",
                        [1107,  588, 11311]]) #  " really like chocolate"]

In [40]:
with torch.no_grad():
    logits=model(inputs)
probas=torch.softmax(logits,dim=-1)
print(probas.shape)

torch.Size([2, 3, 50257])


In [42]:
token_ids=torch.argmax(probas,dim=-1,keepdim=True)
print(token_ids.shape)

torch.Size([2, 3, 1])


In [43]:
token_ids

tensor([[[16657],
         [  339],
         [42826]],

        [[49906],
         [29669],
         [41751]]])

In [44]:
print(f"Targets batch 1: {token_ids_to_text(targets[0], tokenizer)}")
print(f"Outputs batch 1: {token_ids_to_text(token_ids[0].flatten(), tokenizer)}")

Targets batch 1:  effort moves you
Outputs batch 1:  Armed heNetflix


> model training aims to increase the softmax probablity in the index
positins corresponding to the correct target token ids

### lets see the initial softmax prob scores corresponding ot the target tokens 

>basically what the probs given by our untrained model for the actual target tokens

In [46]:
probas.shape

torch.Size([2, 3, 50257])

In [48]:
targets[0]

tensor([3626, 6100,  345])

In [49]:
targets[1]

tensor([ 1107,   588, 11311])

In [50]:
targets

tensor([[ 3626,  6100,   345],
        [ 1107,   588, 11311]])

In [54]:
text_idx=0
target_probas_1=probas[text_idx,[0,1,2],targets[text_idx]]
text_idx=1
target_probas_2=probas[text_idx,[0,1,2],targets[text_idx]]

In [55]:
print("Text 1:", target_probas_1)

# text_idx = 1
# target_probas_2 = probas[text_idx, [0, 1, 2], targets[text_idx]]
print("Text 2:", target_probas_2)

Text 1: tensor([7.4540e-05, 3.1061e-05, 1.1563e-05])
Text 2: tensor([1.0337e-05, 5.6776e-05, 4.7559e-06])


> we want to maximize all these values as these are the probs of the target tokens that 
are being output by our untrained llm. if we max these, then we will get coherent text

## backprop

>Backpropagation requires a loss function, which calculates the difference
between the model's predicted output (here, the probabilities corresponding to
the target token IDs) and the actual desired output. This loss function
measures how far off the model's predictions are from the target values.

`minimizing the log probs is easier`

In [58]:
log_probs=torch.log(torch.cat((target_probas_1,target_probas_2)))
print(log_probs)

tensor([ -9.5042, -10.3796, -11.3677, -11.4798,  -9.7764, -12.2561])


## average log probability

In [59]:
avg_log_probas=torch.mean(log_probs)
print(avg_log_probas)

tensor(-10.7940)


`main goal: maximize the average log prob => optiimize the model`
`since its log, max pos value is 0`

>The goal is to get the average log probability as close to 0 as possible by
updating the model's weights as part of the training process.

```
However, in deep learning, the common practice isn't to push the average log
probability up to 0 but rather to bring the negative average log probability
down to 0. The negative average log probability is simply the average log
probability multiplied by -1,```

## cross-entropy loss

In [61]:
neg_avg_log_probas = avg_log_probas * -1
print(neg_avg_log_probas)

tensor(10.7940)


In [62]:
logits.shape, targets.shape

(torch.Size([2, 3, 50257]), torch.Size([2, 3]))

For the cross entropy_loss function in PyTorch, we want to flatten these
tensors by combining them over the batch dimension:

In [63]:
logits_flat = logits.flatten(0, 1)
targets_flat = targets.flatten()
print("Flattened logits:", logits_flat.shape)
print("Flattened targets:", targets_flat.shape)

Flattened logits: torch.Size([6, 50257])
Flattened targets: torch.Size([6])


In [64]:
targets_flat

tensor([ 3626,  6100,   345,  1107,   588, 11311])

In [65]:
logits_flat

tensor([[ 0.1113, -0.1057, -0.3666,  ...,  0.2843, -0.8824,  0.1074],
        [-0.6109, -0.5167, -0.7613,  ...,  0.5450, -1.0319, -0.2175],
        [ 0.5707, -0.6459, -0.0701,  ...,  0.7419, -0.1806, -0.2217],
        [-0.2968,  0.1949, -0.1649,  ..., -0.4867,  0.7218, -0.1714],
        [-0.8375,  0.0612, -0.4641,  ...,  0.2327, -0.3889, -0.0770],
        [ 0.5614,  0.6919,  0.8915,  ..., -0.9472,  1.2411, -0.2056]])

In [66]:
loss=torch.nn.functional.cross_entropy(logits_flat,targets_flat)
print(loss)

tensor(10.7940)


## perplexity

Perplexity is a measure often used alongside cross entropy loss to evaluate
the performance of models in tasks like language modeling. It can provide a
more interpretable way to understand the uncertainty of a model in predicting
the next token in a sequence.

In [67]:
perplexity= torch.exp(loss)

In [68]:
perplexity

tensor(48725.8203)

```
In the given example, this would translate to the model
being unsure about which among 48,725 words or tokens in the vocabulary to
generate as the next token.
```

# training and validation set lossses

In [70]:
import os
import requests

file_path = "the-verdict.txt"
url = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt"

if not os.path.exists(file_path):
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    text_data = response.text
    with open(file_path, "w", encoding="utf-8") as file:
        file.write(text_data)
else:
    with open(file_path, "r", encoding="utf-8") as file:
        text_data = file.read()

In [71]:
text_data

'I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no great surprise to me to hear that, in the height of his glory, he had dropped his painting, married a rich widow, and established himself in a villa on the Riviera. (Though I rather thought it would have been Rome or Florence.)\n\n"The height of his glory"--that was what the women called it. I can hear Mrs. Gideon Thwing--his last Chicago sitter--deploring his unaccountable abdication. "Of course it\'s going to send the value of my picture \'way up; but I don\'t think of that, Mr. Rickham--the loss to Arrt is all I think of." The word, on Mrs. Thwing\'s lips, multiplied its _rs_ as though they were reflected in an endless vista of mirrors. And it was not only the Mrs. Thwings who mourned. Had not the exquisite Hermia Croft, at the last Grafton Gallery show, stopped me before Gisburn\'s "Moon-dancers" to say, with tears in her eyes: "We shall not look upon its like again"?\n\nWell!--even 

In [72]:
print(text_data[:99])

I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


In [73]:
total_characters = len(text_data)
total_tokens = len(tokenizer.encode(text_data))

print("Characters:", total_characters)
print("Tokens:", total_tokens)

Characters: 20479
Tokens: 5145


In [74]:
import torch
from torch.utils.data import Dataset, DataLoader
import tiktoken

In [78]:
class GPTDatasetV1(Dataset):
    def __init__(self,txt, tokenizer, max_length, stride):
        # txt-> total text to build the dataset from
        # tokenizer-> tokenzier to use
        # max_length-> max context length
        # stride-> sliding window size, how many tokens to slide for the new window
        self.tokenizer=tokenizer
        self.input_ids=[]
        self.target_ids=[]
        token_ids=tokenizer.encode(txt)
        for i in range(0, len(token_ids)-max_length, stride):
            input_chunk=token_ids[i:i+max_length]
            target_chunk=token_ids[i+1:i+max_length+1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __getitem__(self,idx):
        return self.input_ids[idx],self.target_ids[idx]
    def __len__(self):
        return len(self.input_ids)

def create_dataloaderv1(txt, batch_size=4,max_length=256,
                        stride=128,shuffle=False,drop_last=True,num_workers=0):
    tokenizer=tiktoken.get_encoding("gpt2")
    dataset=GPTDatasetV1(txt,tokenizer,max_length,stride)
    dataloader=DataLoader(dataset, batch_size=batch_size, shuffle=shuffle,drop_last=drop_last)
    return dataloader

In [79]:
train_ratio = 0.90
split_idx = int(train_ratio * len(text_data))
train_data = text_data[:split_idx]
val_data = text_data[split_idx:]


torch.manual_seed(123)

train_loader = create_dataloaderv1(
    train_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=True,
    shuffle=True,
    num_workers=0
)

val_loader = create_dataloaderv1(
    val_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=False,
    shuffle=False,
    num_workers=0
)

In [81]:
print("Train loader:")
for x, y in train_loader:
    print(x.shape, y.shape)
print("\nValidation loader:")
for x, y in val_loader:
    print(x.shape, y.shape)

Train loader:
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])

Validation loader:
torch.Size([2, 256]) torch.Size([2, 256])


In [84]:
train_tokens = 0
for input_batch, target_batch in train_loader:
    train_tokens += input_batch.numel()

val_tokens = 0
for input_batch, target_batch in val_loader:
    val_tokens += input_batch.numel()

print("Training tokens:", train_tokens)
print("Validation tokens:", val_tokens)
print("All tokens:", train_tokens + val_tokens)

Training tokens: 4608
Validation tokens: 512
All tokens: 5120


## cross entropy loss for a given batch

In [85]:
def calc_loss_batch(input_batch,target_batch, model,device):
    input_batch,target_batch=input_batch.to(device),target_batch.to(device)
    logits=model(input_batch)
    loss=torch.nn.functional.cross_entropy(logits.flatten(0,1),target_batch.flatten())
    return loss

def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.
    if len(data_loader) == 0:
        return float("nan")
    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        # Reduce the number of batches to match the total number of batches in the data loader
        # if num_batches exceeds the number of batches in the data loader
        num_batches = min(num_batches, len(data_loader))
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            total_loss += loss.item()
        else:
            break
    return total_loss / num_batches

In [86]:

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    # Use PyTorch 2.9 or newer for stable mps results
    major, minor = map(int, torch.__version__.split(".")[:2])
    if (major, minor) >= (2, 9):
        device = torch.device("mps")
    else:
        device = torch.device("cpu")
else:
    device = torch.device("cpu")


print(f"Using {device} device.")


model.to(device) # no assignment model = model.to(device) necessary for nn.Module classes


torch.manual_seed(123) # For reproducibility due to the shuffling in the data loader

with torch.no_grad(): # Disable gradient tracking for efficiency because we are not training, yet
    train_loss = calc_loss_loader(train_loader, model, device)
    val_loss = calc_loss_loader(val_loader, model, device)

print("Training loss:", train_loss)
print("Validation loss:", val_loss)

Using cpu device.
Training loss: 10.98758347829183
Validation loss: 10.98110580444336


## training an llm